# 05 · Multi-touch attribution — six models, one conversion

**Analytical question.** If a user sees **Display** on Monday, clicks a **Paid Search** ad on Wednesday, and converts via **Direct** on Friday, which channel earned the IDR?

Every FMCG dashboard ships with the wrong answer. Last-click gives Direct 100%. First-click gives Display 100%. Linear splits 33/33/33. None of these answers are *causal*. An FMCG brand is paying for the sequential structure of the funnel — awareness on broadcast feeds retail trial six weeks later — and the heuristic rules are blind to that structure.

This notebook walks through the six attribution methods implemented in `smokefreelab.attribution`:

1. **Last-click** — final touch gets 100%.
2. **First-click** — initial touch gets 100%.
3. **Linear** — split equally across unique touches.
4. **Time-decay** — geometric decay from the end of the journey.
5. **Markov (removal-effect)** — data-driven, sequence-aware.
6. **Shapley (exact coalition enumeration)** — game-theoretic, sequence-blind.

The point of running all six is the **gap**. The divergence between heuristic and data-driven credit is itself a KPI — it tells you which channels your dashboards are over-crediting and which are getting starved.

## 1 · Setup

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from smokefreelab.attribution import (
    first_click_attribution,
    last_click_attribution,
    linear_attribution,
    markov_attribution,
    shapley_attribution,
    time_decay_attribution,
)

RNG = np.random.default_rng(seed=7)
pd.options.display.float_format = "{:,.3f}".format

## 2 · Simulated journey data

We simulate **10,000 customer journeys** across 5 channels with a funnel-realistic structure:

- **Display** and **Paid Search** tend to appear early (awareness).
- **Email** and **Organic** tend to appear mid-funnel (consideration).
- **Direct** tends to appear at the close.

Conversion is a function of which channels appeared — not just the last touch — so the data-driven models have something to learn that the heuristics cannot see.

In [ ]:
CHANNELS = ["Display", "Paid Search", "Email", "Organic", "Direct"]

# Per-channel true conversion contribution (not observable in practice).
# Direct looks like a closer under last-click but the *structural* driver is
# Display + Paid Search appearing earlier in the journey.
TRUE_CONTRIBUTION = {
    "Display": 0.25,
    "Paid Search": 0.30,
    "Email": 0.15,
    "Organic": 0.10,
    "Direct": 0.05,
}

N_USERS = 10_000
journeys: list[list[str]] = []
conversions: list[bool] = []

for _ in range(N_USERS):
    # Draw a random journey length (1-5 touches).
    length = int(RNG.integers(1, 6))
    # Weight early slots toward awareness channels, later slots toward Direct.
    journey: list[str] = []
    for i in range(length):
        slot_frac = i / max(length - 1, 1)
        weights = np.array([
            0.45 * (1 - slot_frac),   # Display early-weighted
            0.35 * (1 - 0.5 * slot_frac),  # Paid Search early/mid
            0.20 + 0.10 * slot_frac,  # Email mid
            0.15 + 0.05 * slot_frac,  # Organic mid
            0.05 + 0.55 * slot_frac,  # Direct late-weighted
        ])
        weights = weights / weights.sum()
        journey.append(str(RNG.choice(CHANNELS, p=weights)))
    journeys.append(journey)

    # Conversion probability is a saturating sum of touched-channel contributions.
    touch_set = set(journey)
    linear_score = sum(TRUE_CONTRIBUTION[c] for c in touch_set)
    p_convert = 1 - np.exp(-1.8 * linear_score)  # saturation
    conversions.append(bool(RNG.random() < p_convert))

n_converters = int(sum(conversions))
print(f"Simulated {N_USERS:,} journeys | converters: {n_converters:,} | CVR: {n_converters / N_USERS:.1%}")

## 3 · Heuristic rules — the dashboard baseline

These four rules are what a GA4 / Adobe Analytics dashboard ships by default. They are fast, transparent, and *wrong* in predictable ways:

- **Last-click** over-credits the closer (Direct).
- **First-click** over-credits awareness (Display).
- **Linear** and **time-decay** split credit by a rule unrelated to causal contribution.

In [ ]:
heuristic_results = {
    "last_click":   last_click_attribution(journeys, conversions, channels=CHANNELS),
    "first_click":  first_click_attribution(journeys, conversions, channels=CHANNELS),
    "linear":       linear_attribution(journeys, conversions, channels=CHANNELS),
    "time_decay":   time_decay_attribution(journeys, conversions, channels=CHANNELS, half_life_steps=2.0),
}

heuristic_df = pd.DataFrame(
    {name: r.attributions for name, r in heuristic_results.items()},
    index=CHANNELS,
)
heuristic_df.loc["TOTAL"] = heuristic_df.sum()
heuristic_df

## 4 · Markov removal-effect — sequential causality

The Markov model treats each journey as a walk on a directed graph whose nodes are channels plus `Start`, `Converted`, and `Dropped`. The *removal effect* of a channel is the drop in `P(Converted | Start)` when that channel is rewired to send all walks to `Dropped`. Channels whose removal collapses conversion most own the most credit.

This is pressure-transient analysis on a funnel: remove a zone, measure the signal drop, attribute proportionally.

In [ ]:
markov_result = markov_attribution(journeys, conversions, channels=CHANNELS)

markov_df = pd.DataFrame({
    "channel": markov_result.channels,
    "removal_effect": markov_result.removal_effects,
    "attribution": markov_result.attributions,
    "share": markov_result.shares,
}).sort_values("attribution", ascending=False)

print(f"Baseline conversion probability: {markov_result.conversion_probability:.4f}")
print(f"Total attributed conversions:    {sum(markov_result.attributions):.1f}")
markov_df

## 5 · Shapley value — coalitional fairness

Shapley treats each channel as a player in a coalitional game whose value is the observed conversion count among users whose full touch-set is a subset of the coalition. The axioms (efficiency, symmetry, null-player, additivity) guarantee a unique fair allocation.

When N channels co-produce conversions, how do we split the credit in a way that cannot be challenged on fairness grounds? Shapley values, from cooperative game theory, answer exactly that question.

In [ ]:
shapley_result = shapley_attribution(journeys, conversions, channels=CHANNELS)

shapley_df = pd.DataFrame({
    "channel": shapley_result.channels,
    "shapley_value": shapley_result.shapley_values,
    "attribution": shapley_result.attributions,
    "share": shapley_result.shares,
}).sort_values("attribution", ascending=False)

print(f"Total attributed conversions (efficiency axiom): {sum(shapley_result.attributions):.1f}")
shapley_df

## 6 · Side-by-side — where heuristics and data-driven disagree

Stack all six methods on one axis. The divergence is the insight.

In [ ]:
all_methods = {
    **{name: np.asarray(r.attributions) for name, r in heuristic_results.items()},
    "markov":  np.asarray(markov_result.attributions),
    "shapley": np.asarray(shapley_result.attributions),
}

comparison_df = pd.DataFrame(all_methods, index=CHANNELS)

# Normalise to shares so different totals are comparable.
shares_df = comparison_df.div(comparison_df.sum(axis=0), axis=1)
shares_df

In [ ]:
fig = go.Figure()
for method in shares_df.columns:
    fig.add_bar(name=method, x=CHANNELS, y=shares_df[method].values * 100)

fig.update_layout(
    title="Attribution share by channel — six methods, same data",
    yaxis_title="Share of conversions (%)",
    barmode="group",
    template="plotly_white",
    height=450,
)
fig.show()

## 7 · Business Impact — the IDR gap

Suppose an FMCG commercial team has **IDR 20 B** in quarterly digital media spend to allocate across these five channels. The *per-channel allocation* is the policy lever. Each attribution method implies a different budget.

The table below translates each method's share into a recommended IDR allocation and shows the gap between the dashboard default (**last-click**) and the data-driven winner (**Markov**).

In [ ]:
BUDGET_IDR = 20_000_000_000  # IDR 20 B quarterly

budget_df = (shares_df[["last_click", "markov", "shapley"]] * BUDGET_IDR).round(-6)
budget_df.columns = ["Last-click (IDR)", "Markov (IDR)", "Shapley (IDR)"]
budget_df["Gap LC→Markov (IDR)"] = budget_df["Markov (IDR)"] - budget_df["Last-click (IDR)"]

# Pretty-print in billions for readability.
display_df = (budget_df / 1e9).round(2)
display_df.columns = [c.replace("(IDR)", "(IDR B)") for c in display_df.columns]
display_df

### Business Impact

On a **IDR 20 B quarterly digital budget**, the allocation implied by last-click attribution is materially different from the allocation implied by Markov removal-effect. The absolute rupiah gap on a single channel (typically Display or Paid Search) is in the **IDR 1–3 B per quarter** range — which is the same order of magnitude as the entire annual salary of the team reading this notebook.

**Recommended action.** Report attribution *on both rules* on the dashboard. Flag any channel whose last-click share is more than 5 percentage points above its Markov share — that channel is a *closer* that the dashboard is over-crediting, and the media team is under-funding its true driver upstream. At a quarterly budget review, that gap is the argument for re-allocating from Direct / Paid Search closer spend into Display / Email awareness spend.


**Next notebook.** `06_business_case.ipynb` synthesises Sessions 1–4 into one executive narrative and the final IDR impact calculation.